# GPU-accelerated BESS optimization: Pyomo + cuOpt vs Pyomo + HiGHS

Perfect-foresight price-arbitrage LP for a battery at **ERCOT North Hub (`HB_NORTH`)**, using **real ERCOT 15-minute real-time prices** (2024-2025) plus simulated battery derates, outages and capacity fade.
The same Pyomo model is solved on CPU (HiGHS) and on GPU (NVIDIA cuOpt), and the solve times are compared across horizon lengths.

**Before running:** `Runtime -> Change runtime type -> GPU` (a T4 is enough).

Honest expectation: for small LPs, HiGHS (a simplex/IPM solver) is typically faster. cuOpt's GPU PDLP method is built for large LPs, so look for where the curves cross as the horizon grows, not for a win at every size.

## 1. Check the GPU and install

In [ ]:
import re, subprocess, sys

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout if smi.returncode == 0 else "NO GPU FOUND - switch the runtime to GPU")
m = re.search(r"CUDA Version:\s*(\d+)", smi.stdout)
cuda_major = int(m.group(1)) if m else None
print("CUDA major:", cuda_major, "| Python:", sys.version.split()[0])

In [ ]:
# uv installs much faster than pip. Pyomo >= 6.10 ships the `cuopt` solver plugin;
# cuOpt wheels need Python 3.11+ and Linux. pyarrow reads the Parquet data files.
%pip install -q uv
!uv pip install --system -q "pyomo>=6.10" highspy pyarrow

if cuda_major == 13:
    !uv pip install --system -q cuopt-cu13
else:  # CUDA 12.x
    !uv pip install --system -q "nvidia-cuda-runtime-cu12==12.9.*" cuopt-cu12

If pip reports dependency conflicts or the next cell fails on import, use `Runtime -> Restart session` and re-run from the top (skip nothing: the installs are quick on the second pass).

## 2. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/andrewjeffallen/accelerated-bess-optimization.git"
if not os.path.exists("bess_opt.py"):
    !git clone -q {REPO_URL}
    %cd accelerated-bess-optimization

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bess_opt as bo, benchmark as bm

print("cuOpt solver available to Pyomo:", bo.cuopt_available())

## 3. Data

`SOURCE = "real"` (default) loads the Parquet files committed in `data/`:

- **Prices (real):** ERCOT `HB_NORTH` real-time 15-minute settlement point prices, 2024-01-01 to 2025-12-31 (731 days, 70,176 intervals), from the ERCOT Public API.
- **Conditions (simulated):** per-interval available power (thermal derates + random outages) and usable energy (capacity fade). **Not measured from a real battery.** Set `USE_CONDITIONS = False` to run with the battery always at rated power and nameplate energy.

`SOURCE = "synthetic"` generates an ERCOT-*like* series (solar dip, evening peak, spikes, negative prices) of any length. **It is not real data.** The earlier Colab results were produced this way.

See the README's *Dataset overview* for details.

In [ ]:
SOURCE = "real"                           # "real" | "synthetic"
USE_CONDITIONS = True                     # real only: simulated derates / outages / fade
DAY_COUNTS = [1, 7, 30, 90, 365, 730]     # horizons to benchmark (real data has 731 days)
INTERVAL_MIN = 15                         # synthetic only

prices, conditions = bm.load_inputs(SOURCE, max(DAY_COUNTS), USE_CONDITIONS, INTERVAL_MIN)

print(f"{prices.name}: {len(prices):,} intervals, dt = {bo.infer_dt_hours(prices)*60:.0f} min, "
      f"{prices.index[0]:%Y-%m-%d} to {prices.index[-1]:%Y-%m-%d}")
print("conditions:", "simulated derates/outages/fade" if conditions is not None else "none (rated limits)")
print(f"price range: {prices.min():,.2f} to {prices.max():,.2f} $/MWh, mean {prices.mean():,.2f}")
prices.iloc[: 7 * 96].plot(figsize=(12, 3), title="First week of prices ($/MWh)");

## 4. Battery and correctness check

Solve one week on both solvers first. The objectives should agree closely (cuOpt's PDLP is a first-order method, so expect agreement to roughly its tolerance, not to the last digit).

In [ ]:
# NOTE: the simulated conditions were generated for the default 100 MW / 200 MWh battery.
# If you change `spec`, set USE_CONDITIONS = False above or regenerate them with
# `python make_data.py telemetry`.
spec = bo.BatterySpec(power_mw=100, energy_mwh=200)   # 2-hour, ~85% round-trip
print(spec)

# Extra cuOpt settings go here, e.g. {"time_limit": 300}. Parameter names are
# forwarded to cuOpt's SolverSettings.set_parameter(); see the cuOpt docs for the list.
GPU_OPTIONS = {}
CPU_OPTIONS = {}

dt = bo.infer_dt_hours(prices)
n_week = int(7 * 24 / dt)
week = prices.iloc[:n_week]
week_cond = None if conditions is None else conditions.iloc[:n_week]
bo.warmup(bo.CPU_SOLVER); bo.warmup(bo.GPU_SOLVER, GPU_OPTIONS)

m_cpu = bo.build_model(week.to_numpy(), dt, spec, week_cond); r_cpu = bo.solve(m_cpu, bo.CPU_SOLVER, CPU_OPTIONS)
m_gpu = bo.build_model(week.to_numpy(), dt, spec, week_cond); r_gpu = bo.solve(m_gpu, bo.GPU_SOLVER, GPU_OPTIONS)
print(r_cpu); print(r_gpu)
print(f"relative objective gap: {abs(r_gpu['objective']-r_cpu['objective'])/abs(r_cpu['objective']):.2e}")

bm.plot_dispatch(week, bo.extract_dispatch(m_gpu), "One week, optimal dispatch (cuOpt)");

## 5. Benchmark

Times are wall-clock for `SolverFactory.solve(...)`, which includes Pyomo handing the model to the solver (for cuOpt, also the host-to-GPU transfer). Pyomo's own model-build time is the same for both and is reported separately. A tiny warm-up solve is run first so CUDA initialization is not billed to the first size.

The largest sizes (730 days at 15 min is about 70k intervals and 210k variables) can take a while on the CPU. Trim `DAY_COUNTS` if you're short on time.

In [ ]:
df = bm.run_benchmark(prices, DAY_COUNTS, spec=spec, cpu_options=CPU_OPTIONS,
                      gpu_options=GPU_OPTIONS, run_gpu=True, repeats=1,
                      conditions=conditions)
summary = bm.summarize(df)
summary

In [ ]:
bm.plot_results(summary, save_path="benchmark_results.png")
summary.to_csv("benchmark_results.csv", index=False)
plt.show()

## Reading the results

- `speedup_x` is CPU time / GPU time: above 1 means the GPU was faster.
- `obj_rel_gap` should be small (about 1e-4 or below). A large gap means cuOpt stopped early or at a looser tolerance, so the timing comparison isn't like-for-like. Check the termination status and tighten the tolerances in `GPU_OPTIONS`.
- To stress the GPU further, increase `DAY_COUNTS`, use `INTERVAL_MIN = 5`, or extend the model (multiple batteries, ancillary services co-optimization, scenarios).